# Day 42: Data Collection & Preprocessing

Build a dataset pipeline for your capstone project (example: a QA dataset from web scraping).

In [ ]:
import os
import json
import pandas as pd
import requests
from bs4 import BeautifulSoup
from datasets import Dataset
from PIL import Image
from pathlib import Path

## 1. Example: Scrape a website for text data

In [ ]:
def scrape_page(url):
    response = requests.get(url)
    soup = BeautifulSoup(response.text, 'html.parser')
    # Remove script and style
    for tag in soup(['script', 'style', 'nav', 'footer']):
        tag.decompose()
    text = soup.get_text(separator='\n', strip=True)
    return text[:1000]  # limit for demo

# Example URLs (replace with your own)
urls = [
    "https://en.wikipedia.org/wiki/Artificial_intelligence",
    "https://en.wikipedia.org/wiki/Machine_learning",
]
data = []
for url in urls:
    text = scrape_page(url)
    data.append({"source": url, "text": text})

df = pd.DataFrame(data)
df.head()

## 2. Clean the text

In [ ]:
import re

def clean_text(text):
    text = re.sub(r'\n+', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    text = text.strip()
    return text

df['cleaned_text'] = df['text'].apply(clean_text)
df[['source', 'cleaned_text']].head()

## 3. Split into chunks (for RAG/long context)

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)

all_chunks = []
for _, row in df.iterrows():
    chunks = splitter.split_text(row['cleaned_text'])
    for chunk in chunks:
        all_chunks.append({"source": row['source'], "chunk": chunk})

chunk_df = pd.DataFrame(all_chunks)
print(f"Created {len(chunk_df)} chunks")
chunk_df.head()

## 4. Save dataset

In [ ]:
# Save as CSV
chunk_df.to_csv("my_dataset.csv", index=False)

# Or as Hugging Face Dataset (recommended)
hf_dataset = Dataset.from_pandas(chunk_df)
hf_dataset.save_to_disk("./my_dataset_hf")

print("Dataset saved.")

## 5. Image preprocessing example (if your project needs images)

In [ ]:
def preprocess_image(image_path, target_size=(512, 512)):
    img = Image.open(image_path).convert('RGB')
    img = img.resize(target_size)
    # Normalise to [0,1] if needed
    img_array = np.array(img) / 255.0
    return img_array

# Example: iterate over image folder
# image_dir = Path("./raw_images")
# for img_path in image_dir.glob("*.jpg"):
#     processed = preprocess_image(img_path)
#     # save or store